# NVIDIA Nemotron on Amazon Bedrock

> **Sample code — not for production.** Provided as AWS Content under the AWS
> Customer Agreement; do not use it in production accounts or on production or other
> critical data. Running these cells calls Amazon Bedrock and incurs charges. Full
> disclaimer in the [README](../README.md#disclaimer).

NVIDIA's Nemotron models, from Nemotron 3 Super (120B), the default here, down to
Nemotron Nano 9B v2. Nano 12B v2 also reads images.

| Model | `bedrock-mantle` ID | `bedrock-runtime` ID | Notes |
|---|---|---|---|
| Nemotron 3 Super 120B | `nvidia.nemotron-super-3-120b` | `nvidia.nemotron-super-3-120b` | Largest; the default here |
| Nemotron 3 Nano 30B | `nvidia.nemotron-nano-3-30b` | `nvidia.nemotron-nano-3-30b` | Mid-size |
| Nemotron Nano 12B v2 | `nvidia.nemotron-nano-12b-v2` | `nvidia.nemotron-nano-12b-v2` | Small; takes images |
| Nemotron Nano 9B v2 | `nvidia.nemotron-nano-9b-v2` | `nvidia.nemotron-nano-9b-v2` | Smallest, lowest latency |

IDs as of September 2026, in `us-east-1`. Which Regions carry each model changes as
models launch, so the setup cell checks your Region.

On `bedrock-mantle` these models serve the OpenAI-compatible **Chat Completions** API at
the bare `/v1` path. On `bedrock-runtime` every model here serves **Converse** under the
same ID (section 8).

**Covered:** first call, streaming, multi-turn, reasoning, tool use, structured output,
vision with Nano 12B v2, and Converse on `bedrock-runtime`.

## Setup

Install the pinned requirements first (`pip install -r ../requirements.txt`, Python 3.11
or later). The notebook calls Bedrock with the OpenAI SDK and boto3. From
`_shared/bedrock.py`, this repository's own helper module, it takes an availability
check and a lenient JSON parser, plus the sample images and their known answers for the
vision cells.

Endpoints, authentication and URL paths are explained in
[`00-foundations/01`](../00-foundations/01-endpoints-auth-and-the-three-paths.ipynb).

In [1]:
import json
import sys

import boto3
from aws_bedrock_token_generator import provide_token
from openai import OpenAI

sys.path.insert(0, "../_shared")
from bedrock import endpoints_for, parse_json_lenient

REGION = "us-east-1"
SUPER = "nvidia.nemotron-super-3-120b"
NANO30 = "nvidia.nemotron-nano-3-30b"
NANO12 = "nvidia.nemotron-nano-12b-v2"
NANO9 = "nvidia.nemotron-nano-9b-v2"

# bedrock-mantle: the OpenAI SDK, with a short-term Bedrock API key minted from your
# AWS credentials. Mint it when you build the client, because the key expires.
client = OpenAI(
    base_url=f"https://bedrock-mantle.{REGION}.api.aws/v1",
    api_key=provide_token(region=REGION),
    timeout=120,
)

# bedrock-runtime: boto3 signs each request with SigV4, so there is no key to mint.
runtime = boto3.client("bedrock-runtime", region_name=REGION)

for model in (SUPER, NANO30, NANO12, NANO9,):
    served = [name for name, ok in endpoints_for(model, REGION).items() if ok]
    print(f"{model:30} served by: {', '.join(served) or 'neither endpoint in ' + REGION}")

nvidia.nemotron-super-3-120b   served by: mantle, runtime
nvidia.nemotron-nano-3-30b     served by: mantle, runtime
nvidia.nemotron-nano-12b-v2    served by: mantle, runtime
nvidia.nemotron-nano-9b-v2     served by: mantle, runtime


## 1. First call

A single request with a short question. If the model spends the whole `max_tokens`
budget before it answers, `content` is `None` and `finish_reason` is `length`, so the
cell prints both.

In [2]:
response = client.chat.completions.create(
    model=SUPER,
    messages=[{"role": "user", "content": "What is the capital of Australia? Answer in one word."}],
    max_tokens=1000,
)
choice = response.choices[0]
print((choice.message.content or "").strip())
print(f"finish_reason={choice.finish_reason}  total_tokens={response.usage.total_tokens}")

Canberra
finish_reason=stop  total_tokens=31


## 2. Streaming

Set `stream=True` and read `delta.content` from each chunk. Check that a chunk has
choices before indexing it, and read `finish_reason` from the last one: `stop` means the
answer is complete.

In [3]:
stream = client.chat.completions.create(
    model=SUPER,
    messages=[{"role": "user", "content": "Count from 1 to 10, separated by commas."}],
    max_tokens=1000,
    stream=True,
)
finish_reason = None
for chunk in stream:
    if chunk.choices and chunk.choices[0].delta.content:
        print(chunk.choices[0].delta.content, end="", flush=True)
    if chunk.choices and chunk.choices[0].finish_reason:
        finish_reason = chunk.choices[0].finish_reason
print(f"\nfinish_reason={finish_reason}")

1, 2, 3, 4, 5, 6, 7, 8, 9, 10


finish_reason=stop


## 3. Multi-turn

Chat Completions keeps no conversation state on the server, so each request carries the
whole history. Append the assistant's reply before the next user turn.

In [4]:
messages = [
    {"role": "system", "content": "You are a concise assistant."},
    {"role": "user", "content": "My favourite colour is teal. Reply with one word: noted."},
]
first = client.chat.completions.create(model=SUPER, messages=messages, max_tokens=1000)
messages.append({"role": "assistant", "content": first.choices[0].message.content})
messages.append({"role": "user", "content": "What is my favourite colour? One word."})
second = client.chat.completions.create(model=SUPER, messages=messages, max_tokens=1000)

print("turn 1:", (first.choices[0].message.content or "").strip())
print("turn 2:", (second.choices[0].message.content or "").strip())

turn 1: noted
turn 2: teal


## 4. Reasoning

Set `reasoning_effort` and Nemotron Super reasons before it answers. The reasoning comes
back in `choices[0].message.reasoning`, a field the OpenAI schema does not define. The
SDK still exposes it as an attribute, but code written against the published types will
not see it. Reasoning is billed as output and spends the budget before the answer
starts, so give it a larger `max_tokens`.

In [5]:
response = client.chat.completions.create(
    model=SUPER,
    messages=[{"role": "user", "content": 'A bat and a ball cost $1.10 in total. The bat costs $1.00 more than the ball. How much does the ball cost? Answer with the amount only.'}],
    reasoning_effort="high",
    max_tokens=4000,
)
message = response.choices[0].message
trace = getattr(message, "reasoning", None) or ""
print("answer   :", (message.content or "").strip())
print(f"reasoning: {len(trace)} characters")
print(f"finish_reason={response.choices[0].finish_reason}  "
      f"completion_tokens={response.usage.completion_tokens}")

answer   : 0.05
reasoning: 1140 characters
finish_reason=stop  completion_tokens=418


## 5. Tool use

A tool loop sends the tool definitions, runs each call the model makes, returns the
results, and repeats until the model answers in text. Check each call's arguments before
running it: `finish_reason == "tool_calls"` means the model wants a tool, and says
nothing about whether the arguments are usable.

In [6]:
def lookup_inventory(sku: str) -> dict:
    """Stand-in for a real inventory service."""
    stock = {"A-100": 42, "B-200": 0}
    if sku.upper() not in stock:
        return {"sku": sku, "error": "unknown SKU"}
    return {"sku": sku, "quantity": stock[sku.upper()]}


TOOLS = [{
    "type": "function",
    "function": {
        "name": "lookup_inventory",
        "description": "Current stock level for a SKU.",
        "parameters": {
            "type": "object",
            "properties": {"sku": {"type": "string", "description": "SKU code, e.g. A-100"}},
            "required": ["sku"],
        },
    },
}]
IMPLEMENTATIONS = {"lookup_inventory": lookup_inventory}


def run_tool(name: str, args: dict) -> dict:
    """Refuse a call that names an unknown tool or does not match the tool's parameters."""
    spec = next((t["function"] for t in TOOLS if t["function"]["name"] == name), None)
    if spec is None:
        return {"error": f"unknown tool {name}"}
    params = spec["parameters"]
    missing = [k for k in params["required"] if k not in args]
    unknown = [k for k in args if k not in params["properties"]]
    wrong_type = [k for k in args if k in params["properties"] and params["properties"][k].get("type") == "string"
                  and not isinstance(args[k], str)]
    if missing or unknown or wrong_type:
        return {"error": f"bad arguments: missing {missing}, unknown {unknown}, not a string {wrong_type}"}
    return IMPLEMENTATIONS[name](**args)

In [7]:
messages = [{"role": "user", "content": "How many units of SKU A-100 do we have?"}]
for _ in range(5):  # bounded, so a model that keeps calling tools cannot loop forever
    response = client.chat.completions.create(
        model=SUPER, messages=messages, tools=TOOLS, max_tokens=1500
    )
    message = response.choices[0].message
    if not message.tool_calls:
        break
    messages.append(message.model_dump(exclude_none=True))
    for call in message.tool_calls:
        args = parse_json_lenient(call.function.arguments)
        result = run_tool(call.function.name, args)
        print(f"tool call: {call.function.name}({args}) -> {result}")
        messages.append({"role": "tool", "tool_call_id": call.id, "content": json.dumps(result)})
else:
    raise RuntimeError("no answer after 5 rounds of tool calls")

print("answer:", (message.content or "").strip())

tool call: lookup_inventory({'sku': 'A-100'}) -> {'sku': 'A-100', 'quantity': 42}


answer: We have **42 units** of SKU A-100 in stock.


## 6. Structured output

`response_format` with a JSON Schema asks the model for output in that shape. Check
`finish_reason` first, because a cut-off answer is cut-off JSON. Parse leniently,
because some models add text after a valid object. A schema controls the shape of the
output and has no say over the values in it, so the cell checks each value against the
source text.

In [8]:
SCHEMA = {
    "type": "object",
    "properties": {
        "invoice_id": {"type": "string"},
        "vendor": {"type": "string"},
        "total": {"type": "number"},
        "currency": {"type": "string"},
    },
    "required": ["invoice_id", "vendor", "total", "currency"],
    "additionalProperties": False,
}
SOURCE = "Invoice: INV-1042\nVendor: Acme Pty Ltd\nAmount due: AUD 1,280.50\nDue date: 30 October 2026"
EXPECTED = {"invoice_id": "INV-1042", "vendor": "Acme Pty Ltd", "total": 1280.5, "currency": "AUD"}

response = client.chat.completions.create(
    model=SUPER,
    messages=[{"role": "user", "content": (
        "Extract invoice_id, vendor, total and currency from this invoice as JSON. Write the "
        f"total as a number without a thousands separator, so 2,345.60 becomes 2345.6.\n\n{SOURCE}")}],
    response_format={"type": "json_schema", "json_schema": {"name": "invoice", "strict": True, "schema": SCHEMA}},
    max_tokens=1500,
)
choice = response.choices[0]
if choice.finish_reason != "stop":
    raise RuntimeError(f"incomplete output (finish_reason={choice.finish_reason}); raise max_tokens")

invoice = parse_json_lenient(choice.message.content)
print(json.dumps(invoice, indent=2))
wrong = {k: invoice.get(k) for k, v in EXPECTED.items() if invoice.get(k) != v}
print("values match the source" if not wrong else f"values that do NOT match the source: {wrong}")

{
  "invoice_id": "INV-1042",
  "vendor": "Acme Pty Ltd",
  "total": 1280.5,
  "currency": "AUD"
}
values match the source


## 7. Vision with Nano 12B v2

Send the image as a data URL in an `image_url` content part, next to the text part. The
slide comes from a public AWS talk and ships with this repository, so the answer is
known: its title is `Ingestion from database`, and the cell counts how many of its three
callout lines the model quotes.

In [9]:
from bedrock import SLIDE_CALLOUTS, SLIDE_TITLE, keyword_recall, slide_data_url

response = client.chat.completions.create(
    model=NANO12,
    messages=[{"role": "user", "content": [
        {"type": "image_url", "image_url": {"url": slide_data_url()}},
        {"type": "text", "text": "Read this slide. Give its title, then quote the three green callout lines."},
    ]}],
    max_tokens=800,
)
text = response.choices[0].message.content or ""
hits, total = keyword_recall(text, SLIDE_CALLOUTS)
print(text.strip()[:500])
print(f"\ntitle found: {SLIDE_TITLE.lower() in text.lower()}   callouts found: {hits}/{total}")

Here is the slide content transcribed:

**Title**: Ingestion from database

*Pay according to job duration.*
*AWS Glue with serverless ETL.*
*Data lake on Amazon S3 with storage tiers*

*Lower compute cost, up to 90%*
*Lower storage cost for rarely accessed data*

title found: True   callouts found: 3/3


## 8. On `bedrock-runtime`: Converse

Every model here is on `bedrock-runtime` under the same ID. Converse has one request
shape for every provider: `content` is a list of blocks, and settings such as the token
limit go in `inferenceConfig`. An option only one provider understands, such as
`reasoning_effort`, goes in `additionalModelRequestFields`, which Converse passes to the
model unchecked. Reasoning arrives as a `reasoningContent` block ahead of the text, so
read blocks by type instead of taking `content[0]`. The Converse tool loop is in
[`00-foundations/04`](../00-foundations/04-bedrock-runtime-converse-and-profiles.ipynb).

In [10]:
response = runtime.converse(
    modelId="nvidia.nemotron-super-3-120b",
    messages=[{"role": "user", "content": [{"text": "What is the capital of Australia? Answer in one word."}]}],
    inferenceConfig={"maxTokens": 2000},
    additionalModelRequestFields={"reasoning_effort": "high"},
)
blocks = response["output"]["message"]["content"]
text = "".join(b["text"] for b in blocks if "text" in b)
print("blocks    :", [next(iter(b)) for b in blocks])
print("answer    :", text.strip())
trace = "".join(b["reasoningContent"].get("reasoningText", {}).get("text", "") for b in blocks if "reasoningContent" in b)
print("reasoning :", len(trace), "characters")
print("stopReason:", response["stopReason"])

blocks    : ['reasoningContent', 'text']
answer    : Canberra
reasoning : 124 characters
stopReason: end_turn


## Things that differ

What trips people up on this family, and what to do about it.

| Behaviour | What you see | What to do |
|---|---|---|
| Responses API | `POST /v1/responses` returns 400 `The model '…' does not support the '/v1/responses' API` | Use Chat Completions. |
| Path prefix | `/openai/v1/chat/completions` returns 400 ``model `…` isn't supported on this route`` | Use the bare `/v1` path. `/openai/v1` is for the hosted OpenAI GPT, Gemma 4 and Grok models. |
| Conversation state | No `previous_response_id` | Send the full `messages` list on every turn. |
| Empty answer | HTTP 200, `content` is `None`, `finish_reason` is `length` | The model spent `max_tokens` before answering, often on reasoning. Raise the budget and check `finish_reason` before using the text. |
| Structured output values | Valid JSON that holds a wrong value | A schema only controls the shape. Check the values you depend on. |
| Rate limits | 429 under load | Throttling is token-based and there is no requests-per-minute quota. Retry with backoff ([`00-foundations/03`](../00-foundations/03-scaling-tiers-and-latency.ipynb)). |
| Metrics | Nothing under `AWS/Bedrock` in CloudWatch | Mantle metrics are in the `AWS/BedrockMantle` namespace. |
| Images | Only Nano 12B v2 reads images in this family | Route image requests to it (section 7). |

## Next

- Other vision models: [`04-qwen/02`](../04-qwen/02-qwen3-coder-and-vision.ipynb)
  (Qwen3-VL), [`12-writer-palmyra/01`](../12-writer-palmyra/01-palmyra-vision.ipynb)
  (Palmyra Vision)
- Other families on Chat Completions:
  [`04-qwen/01`](../04-qwen/01-qwen3-core-and-tools.ipynb),
  [`07-mistral/01`](../07-mistral/01-mistral-text-and-sizes.ipynb),
  [`05-deepseek/01`](../05-deepseek/01-deepseek-v3-reasoning.ipynb)
- Choosing a model and API across families:
  [`99-cross-cutting/01`](../99-cross-cutting/01-choosing-a-model-and-api.ipynb)
- Quotas, retries and latency:
  [`00-foundations/03`](../00-foundations/03-scaling-tiers-and-latency.ipynb)
- Production hardening:
  [`99-cross-cutting/03`](../99-cross-cutting/03-production-hardening.ipynb)